# Harmonização do SIOSE AR 

Primeiro filtra e junta Badajoz e Cáceres; depois harmoniza os atributos diretamente com SQLite.

## 1. Configuração

In [ ]:
from pathlib import Path
import shutil
import sqlite3
import subprocess

import pandas as pd
import pyogrio

area = "extremadura"
base = f"/code/data/processed/{area}"
raw_lulc = "/code/data/raw/lulc"
aoi_file = f"{base}/aoi/{area}.shp"
harm_file = f"{base}/lulc/harmonized_tables/lookup_lulc_harmonizacao.csv"
out_harm = f"{base}/lulc/harmonized_vectors"

source_layer = "t_poligonos"
code_field = "ID_COBERTURA_MAX"


years = [2020]
overwrite = True

def first_existing(*paths):
    for path in paths:
        if Path(path).exists():
            return path
    return paths[0]

siose_ar = {
    2017: [
        f"{raw_lulc}/SIOSE_2017_Badajoz.gpkg",
        f"{raw_lulc}/SIOSE_2017_Caceres.gpkg",
    ],
    2020: [
        f"{raw_lulc}/SIOSE_2020_Badajoz.gpkg",
        first_existing(
            f"{raw_lulc}/SIOSE_2020_Caceres.gpkg",
            f"{raw_lulc}/SIOSE2020_Caceres.gpkg",
        ),
    ],
}

Path(out_harm).mkdir(parents=True, exist_ok=True)

print("Área:", area)
print("Anos:", years)
print("Origem:", raw_lulc)
print("Destino:", out_harm)

## 2. Verificação leve dos inputs

In [ ]:
for command in ["ogr2ogr", "ogrinfo"]:
    if shutil.which(command) is None:
        raise RuntimeError(f"O comando {command} não está disponível.")

required_files = [aoi_file, harm_file]
for year in years:
    required_files.extend(siose_ar[year])

missing_files = [
    file for file in required_files
    if not Path(file).exists()
]

if missing_files:
    raise FileNotFoundError("Ficheiros em falta:\n" + "\n".join(missing_files))

for year in years:
    for file in siose_ar[year]:
        layers = pyogrio.list_layers(file)[:, 0].tolist()
        if source_layer not in layers:
            raise ValueError(
                f"A layer '{source_layer}' não existe em {file}."
            )

        info = pyogrio.read_info(file, layer=source_layer)
        if code_field not in info['fields']:
            raise ValueError(
                f"O campo '{code_field}' não existe em {file}."
            )

        print(
            year, '|', Path(file).name,
            '| feições:', info['features'],
            '| CRS:', info['crs'],
            '| estado: ok',
        )

## 3. Tabela de harmonização

In [ ]:
harm = pd.read_csv(
    harm_file,
    dtype={
        "product": str,
        "code_original": str,
        "label_harmonizada": str,
        "incluir_modelo": int,
        "obs": str,
        "id_harm": "Int64",
    },
)

lookup = harm.loc[
    (harm["product"] == "siose_ar")
    & (harm["incluir_modelo"] == 1)
    & harm["id_harm"].notna(),
    ["code_original", "id_harm", "label_harmonizada"],
].copy()

lookup["code_original"] = (
    lookup["code_original"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

lookup["label_harmonizada"] = (
    lookup["label_harmonizada"]
    .fillna("")
    .astype(str)
)

lookup = (
    lookup
    .drop_duplicates(subset="code_original")
    .sort_values("code_original")
    .reset_index(drop=True)
)

if lookup.empty:
    raise ValueError("Não existem correspondências válidas para siose_ar.")

valid_codes = [int(code) for code in lookup["code_original"]]
where_codes = (
    f'"{code_field}" IN ('
    + ", ".join(str(code) for code in valid_codes)
    + ")"
)

print("Códigos incluídos:", len(lookup))
print("Filtro:", where_codes)
display(lookup.head(10))

## 4. Funções auxiliares

In [ ]:
def remove_output(file):
    for suffix in ["", "-wal", "-shm", ".aux.xml"]:
        Path(file + suffix).unlink(missing_ok=True)

def run_command(command):
    print('Executar:', ' '.join(command[:8]), '...')
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    if result.stdout:
        print(result.stdout[-4000:])

    if result.returncode != 0:
        raise RuntimeError(
            f"O comando terminou com o código {result.returncode}."
        )

## 5. Cópia e união dos vetores

Esta etapa filtra as classes relevantes, mantém apenas `ID_COBERTURA_MAX` e junta Badajoz e Cáceres. A cache SQLite fica limitada a **64 MB**.

In [ ]:
target_crs = pyogrio.read_info(
    aoi_file
)["crs"]


if target_crs is None:
    raise ValueError(
        "A área de estudo não tem CRS."
    )


outputs = {}


for year in years:
    output_file = (
        f"{out_harm}/"
        f"siose_ar_{year}_harm.gpkg"
    )

    output_layer = (
        f"siose_ar_{year}_harm"
    )

    if overwrite:
        remove_output(output_file)
    elif Path(output_file).exists():
        raise FileExistsError(
            output_file
        )

    print(
        f"\n{'=' * 60}\n"
        f"SIOSE AR {year}\n"
        f"{'=' * 60}"
    )

    for index, file in enumerate(
        siose_ar[year]
    ):
        info = pyogrio.read_info(
            file,
            layer=source_layer,
        )

        geometry_name = info.get(
            "geometry_name"
        )

        if not geometry_name:
            raise ValueError(
                f"Não foi possível identificar "
                f"o campo geométrico de {file}."
            )

        sql = f'''
            SELECT
                "{geometry_name}",
                "{code_field}"
            FROM "{source_layer}"
            WHERE {where_codes}
        '''

        command = [
            "ogr2ogr",
            "--config",
            "OGR_SQLITE_CACHE",
            "64",
            "-f",
            "GPKG",
        ]

        if index > 0:
            command.extend([
                "-update",
                "-append",
            ])

        command.extend([
            output_file,
            file,
            "-nln",
            output_layer,
            "-dialect",
            "SQLite",
            "-sql",
            sql,
            "-nlt",
            "PROMOTE_TO_MULTI",
            "-gt",
            "10000",
        ])

        if info["crs"] != target_crs:
            command.extend([
                "-t_srs",
                target_crs,
            ])

        if index == 0:
            command.extend([
                "-lco",
                "GEOMETRY_NAME=geom",
                "-lco",
                "SPATIAL_INDEX=NO",
            ])

        print(
            f"\nProcessar {index + 1}/"
            f"{len(siose_ar[year])}: "
            f"{Path(file).name}"
        )

        run_command(command)

    outputs[year] = {
        "file": output_file,
        "layer": output_layer,
    }

    info = pyogrio.read_info(
        output_file,
        layer=output_layer,
    )

    print(
        "Feições reunidas:",
        info["features"],
    )


## 6. Harmonização dos atributos com SQLite

A reclassificação é feita apenas depois da cópia vetorial.

In [ ]:
for year, cfg in outputs.items():
    file = cfg["file"]
    layer = cfg["layer"]

    print(f"\nHarmonizar atributos de {year}...")

    with sqlite3.connect(file) as conn:
        conn.execute("PRAGMA cache_size = -65536;")
        conn.execute("PRAGMA temp_store = FILE;")
        conn.execute("PRAGMA synchronous = NORMAL;")

        columns = {
            row[1]
            for row in conn.execute(
                f'PRAGMA table_info("{layer}");'
            )
        }

        if "code_original" not in columns:
            conn.execute(
                f'ALTER TABLE "{layer}" ADD COLUMN code_original TEXT;'
            )

        if "id_harm" not in columns:
            conn.execute(
                f'ALTER TABLE "{layer}" ADD COLUMN id_harm INTEGER;'
            )

        if "classe_harm" not in columns:
            conn.execute(
                f'ALTER TABLE "{layer}" ADD COLUMN classe_harm TEXT;'
            )

        conn.execute("DROP TABLE IF EXISTS lookup_siose_ar;")
        conn.execute(
            "CREATE TABLE lookup_siose_ar ("
            "code_original TEXT PRIMARY KEY, "
            "id_harm INTEGER NOT NULL, "
            "classe_harm TEXT NOT NULL);"
        )

        conn.executemany(
            "INSERT INTO lookup_siose_ar "
            "(code_original, id_harm, classe_harm) "
            "VALUES (?, ?, ?);",
            [
                (
                    str(row.code_original),
                    int(row.id_harm),
                    str(row.label_harmonizada),
                )
                for row in lookup.itertuples()
            ],
        )

        conn.execute(
            f'UPDATE "{layer}" SET code_original = '
            f'TRIM(CAST("{code_field}" AS TEXT));'
        )

        conn.execute(
            f'UPDATE "{layer}" SET '
            "id_harm = ("
            "SELECT id_harm FROM lookup_siose_ar "
            f'WHERE lookup_siose_ar.code_original = "{layer}".code_original), '
            "classe_harm = ("
            "SELECT classe_harm FROM lookup_siose_ar "
            f'WHERE lookup_siose_ar.code_original = "{layer}".code_original);'
        )

        missing = conn.execute(
            f'SELECT COUNT(*) FROM "{layer}" WHERE id_harm IS NULL;'
        ).fetchone()[0]

        if missing:
            raise ValueError(f"Existem {missing} feições sem correspondência.")

        conn.execute("DROP TABLE lookup_siose_ar;")
        conn.commit()

    print("Atributos harmonizados.")

## 7. Índice espacial

In [ ]:
for year, cfg in outputs.items():
    print(f"Criar índice espacial de {year}...")

    run_command([
        "ogrinfo",
        "-q",
        cfg["file"],
        "-dialect",
        "SQLite",
        "-sql",
        (
            "SELECT CreateSpatialIndex"
            f"('{cfg['layer']}', 'geom')"
        ),
    ])

## 8. Verificação final

In [ ]:
for year, cfg in outputs.items():
    file = cfg["file"]
    layer = cfg["layer"]

    info = pyogrio.read_info(file, layer=layer)

    with sqlite3.connect(f"file:{file}?mode=ro", uri=True) as conn:
        check = conn.execute("PRAGMA quick_check(1);").fetchall()

        counts = pd.read_sql_query(
            (
                f'SELECT id_harm, classe_harm, COUNT(*) AS n_feicoes '
                f'FROM "{layer}" '
                "GROUP BY id_harm, classe_harm "
                "ORDER BY id_harm;"
            ),
            conn,
        )

        total = conn.execute(
            f'SELECT COUNT(*) FROM "{layer}";'
        ).fetchone()[0]

        missing = conn.execute(
            f'SELECT COUNT(*) FROM "{layer}" WHERE id_harm IS NULL;'
        ).fetchone()[0]

    print(f"\nSIOSE AR {year}")
    print("Ficheiro:", file)
    print("Layer:", layer)
    print("Feições:", total)
    print("CRS:", info["crs"])
    print("Geometria:", info["geometry_type"])
    print("Integridade:", check)
    print("Sem correspondência:", missing)
    display(counts)